# Stochastic kinetics: trajectories, densities and first passage

**Goal:** predict a transition time from a free-energy landscape and diffusion, then test the prediction. Use dimensionless units with $k_BT=1$, $D=1$. Run cells in order. Exercise prompts ask you to modify or extend runnable starter code. For full derivations see `notes/derivations.md`.

**Learning outcomes:** explain the Itô SDE and forward equation; interpret boundary conditions; estimate a first-passage time and compare independent methods; state limitations for molecular coordinates.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid, trapezoid
from stochkin import potential, force, mfpt_quadrature, grid_generator, grid_mfpt, simulate_fpt
rng = np.random.default_rng(123)
plt.rcParams.update({"figure.figsize": (7, 4), "font.size": 11})

ModuleNotFoundError: No module named 'stochkin'

## 1. Diffusion: an ensemble of random paths

For $dX_t=\sqrt{2D}\,dW_t$, $X_0=0$, Gaussian increments obey $\Delta W\sim N(0,\Delta t)$. Predict the variance at time $t$ before running the cell.

**Task 1:** plot $\mathrm{Var}(X_t)$ alongside its analytic value. Double $D$. How do single paths compare with the ensemble mean?

In [ ]:
D, dt, nsteps, npaths = 1.0, 0.01, 400, 3000
noise = rng.normal(size=(npaths, nsteps))
paths = np.column_stack([np.zeros(npaths), np.cumsum(np.sqrt(2*D*dt)*noise, axis=1)])
t = np.arange(nsteps+1)*dt
plt.plot(t, paths[:10].T, alpha=.6)
plt.xlabel('time'); plt.ylabel('position'); plt.show()
# TODO: compare paths.var(axis=0) with 2*D*t.

## 2. From an SDE to Fokker–Planck

For $dX=a(X)dt+\sqrt{2D(X)}dW$ and smooth $f$, Itô's formula gives $d\mathbb E[f]/dt=\mathbb E[af'+Df'']$. Integrate by parts to obtain $\partial_t p=-\partial_x(ap)+\partial_x^2(Dp)=-\partial_x J$.

**Task 2:** write the current $J$. What must hold at a reflecting boundary? Show that $p_\mathrm{eq}\propto e^{-\beta F}$ has zero current for constant $D$ and $a=-D\beta F'$. Why can two systems with the same $F$ have different kinetics?

## 3. Double well and equilibrium

Use $F(x)=H(x^2-1)^2$ and Euler–Maruyama: $x_{n+1}=x_n+D\beta[-F'(x_n)]\Delta t+\sqrt{2D\Delta t}\xi_n$.

**Task 3:** plot the potential and identify minima and barrier. Make a histogram from the trajectory after burn-in and overlay the normalized Boltzmann curve. Repeat for another $D$ and consider finite sampling.

In [ ]:
H, beta, D, dt = 2.0, 1.0, 1.0, 0.002
x = np.empty(150_000); x[0] = -1.0
for i in range(1, len(x)):
    x[i] = x[i-1] + D*beta*force(x[i-1], H)*dt + np.sqrt(2*D*dt)*rng.normal()
grid = np.linspace(-2, 2, 500)
plt.plot(grid, potential(grid, H)); plt.xlabel('x'); plt.ylabel('F / kBT'); plt.show()
# TODO: histogram x[20000:] and compare with exp(-beta*F), normalized over grid.

## 4. First-passage trajectories

Let $a=-2$ reflect and $b=1$ absorb, with $X_0=-1$. Define $\tau_b=\inf\{t\ge0:X_t\ge b\}$. Predict how $\langle\tau_b\rangle$ changes if $D$ doubles.

**Task 4:** estimate the empirical survival probability $S(t)$ and calculate its integral up to the observation horizon. Report the fraction of censored trajectories and explain why dropping them biases a full-MFPT estimate.

In [ ]:
times, hit = simulate_fpt(n_traj=1200, dt=0.003, max_time=120.0, seed=23)
print('fraction reaching target:', hit.mean())
tgrid = np.linspace(0, 120, 301)
# This indicator includes paths censored at the horizon until the horizon.
survival = np.array([np.mean(times > t) for t in tgrid])
plt.step(tgrid, survival, where='post'); plt.xlabel('time'); plt.ylabel('survival'); plt.show()
# TODO: calculate the sample restricted mean np.minimum(times, 120).mean().

## 5. Backward equation and quadrature

From $T(x)=dt+\mathbb E_x[T(X_{dt})]$, derive $\mathcal LT=-1$. At a reflecting wall $T'(a)=0$; at the absorbing target $T(b)=0$. For the equilibrium model,

$$T(x)=\int_x^b dy\,\frac{e^{\beta F(y)}}{D(y)}\int_a^y dz\,e^{-\beta F(z)}.$$

**Task 5:** derive this result by an integrating factor. Check it at $F=0$ against $[(b-a)^2-(x-a)^2]/(2D)$. Evaluate it for the double well and compare with simulations. Explain the initial distribution and boundaries used.

In [ ]:
a, x0, b = -2.0, -1.0, 1.0
prediction = mfpt_quadrature(x0, a, b)
print('backward equation MFPT:', prediction)
print('fraction observed:', hit.mean())
print('restricted mean:', times.mean())
# TODO: compare with a smaller dt; use new independent random seed and report sampling uncertainty.

## 6. Discrete generator and kinetics

On grid nodes define $Q_{i,i+1}=D h^{-2}e^{-\beta(F_{i+1}-F_i)/2}$ and reverse jumps analogously. The diagonal makes each row sum to zero. The row probability evolves as $\dot p=pQ$; observables evolve under $Q$. With the target removed, solve $-Q_{SS}T=1$.

**Task 6:** compare grid and integral MFPTs; refine the grid. Why do row sums vanish for reflecting boundaries? Check detailed balance $\pi_iQ_{ij}=\pi_jQ_{ji}$.

In [ ]:
nodes = np.linspace(a, b, 101)
Q = grid_generator(nodes)
Tgrid = grid_mfpt(Q)
print('grid MFPT:', np.interp(x0, nodes, Tgrid))
print('quadrature MFPT:', prediction)
# TODO: repeat with 201 or 401 nodes; check row sums and detailed balance.

## 7. Molecular connection and optional committor

For a molecular coordinate $s(\mathbf R)$, $F(s)=-k_BT\ln p_\mathrm{eq}(s)+C$. Projection does not guarantee Markovian dynamics or a known $D(s)$.

**Discussion:** How would you choose molecular reactant and product sets? Does first hitting the barrier necessarily mean reaching product? Why does a well-converged free-energy profile alone not determine a rate?

**Extension:** derive $\mathcal L q=0$ with $q(a)=0$, $q(b)=1$. Compare $q$ with a threshold on a proposed reaction coordinate. See `notes/derivations.md` for variable diffusion, survival and limitations.